In [8]:
import os
from pypdf import PdfReader

manual_path = r"E:\Applied AI - iti\Car Dashboard Detection Project\data\manual\Toyota-Corolla_2026_EN__b4f47a2baa.pdf"
reader = PdfReader(manual_path)

total_pages = len(reader.pages)

print(f"Total pages in manual: {total_pages}")

sample_text = ""
for i in range(min(5, total_pages)):
    sample_text += f"--- PAGE {i+1} ---\n" + reader.pages[i].extract_text()

print(sample_text[:1000])  # Display the first 1000 characters

Total pages in manual: 497
--- PAGE 1 ---
C
1
M
E
J
S
K
L
U
D
R
2
3
4
5
6
7
8
9
0--- PAGE 2 ---
--- PAGE 3 ---
•• PROTECTED 関係者外秘 PROTÉGÉ 
NOTICE TO QUEBEC CONSUMERS: 
Toyota Canada Inc. is pleased to offer a 36 month / 60, 000 k m 
(whichever occurs ﬁrst) limited ne w -vehicle warranty on your 
vehicle. This limited warranty covers repairs on any part of the 
vehicle supplied by Toyota Canada that is defective in material 
or workmanship, subject to certain exceptions that are stated 
in your warranty manual. For more details of the warranty, 
check the Owner’s Manual Supplement included with your 
vehicle, or visit toyota.ca.  
Quebec’s Consumer Protection Act requires Toyota Canada to 
disclose whether we guarantee the availability of replacement 
parts, repair services, and information necessary to maintain 
or repair the vehicle or its components. While Toyota Canada is 
pleased to honour the terms of our limited warranty, please 
note that Toyota in no way guarantees the availabi

### 2.1 Inspection Summary
- **Total Documents:** 1 PDF (`Ford_F150_Manual.pdf` / `Toyota_Corolla_Manual.pdf`)
- **Total Pages:** 497 pages
- **Parsing Status:** Text extracted successfully via `pypdf`. Pages 1–2 contain cover artwork/codes, while standard text begins on Page 3. No OCR needed.

In [9]:
from pypdf import PdfReader

# 1. Load all text from the manual
reader = PdfReader(manual_path)
documents = []

for idx, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():  # Filter out empty pages
        documents.append({"page": idx + 1, "text": text})

print(f"Extracted non-empty pages: {len(documents)}")


# 2. Chunking Function (Fixed-size with overlap)
def chunk_text(docs, chunk_size=1000, overlap=100):
    chunks = []
    for doc in docs:
        page_num = doc["page"]
        text = doc["text"]

        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk_str = text[start:end]

            chunks.append(
                {
                    "text": chunk_str,
                    "metadata": {
                        "source": f"Manual - Page {page_num}",
                        "page": page_num,
                    },
                }
            )

            start += chunk_size - overlap
    return chunks


# 3. Create chunks
all_chunks = chunk_text(documents, chunk_size=1000, overlap=100)
print(f"Total Chunks Created: {len(all_chunks)}")
print("\nSample Chunk 1:\n", all_chunks[50]["text"][:300])

Extracted non-empty pages: 495
Total Chunks Created: 1088

Sample Chunk 1:
 23
1
1For safety and security
For safety and security
.
1-1. For safe use
Before driving................. 24
For safe driving ..............25
Seat belts .......................27
SRS airbags................... 31
Front passenger occupant 
classification system ....41
Exhaust gas precautions
.......


### Chunking Justification
- **Chunk Size (1000 chars):** Captures complete safety instructions, warning descriptions, and troubleshooting steps without splitting critical sentence context.
- **Overlap (100 chars):** Ensures continuity between adjacent chunks so no key details are lost at boundaries.

In [10]:
import os
import chromadb
from chromadb.utils import embedding_functions

# 1. Initialize SentenceTransformer embedding function
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# 2. Set directory to persist Chroma vector store (backend/data/vector_store)
vector_db_path = os.path.abspath("../backend/data/vector_store")
os.makedirs(vector_db_path, exist_ok=True)

# 3. Create persistent client
chroma_client = chromadb.PersistentClient(path=vector_db_path)

# 4. Create or get collection
collection = chroma_client.get_or_create_collection(
    name="car_manual", embedding_function=embedding_fn
)

# 5. Prepare data for batch insertion
ids = [f"chunk_{i}" for i in range(len(all_chunks))]
documents = [c["text"] for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]

# Add in batches of 200 to prevent memory overhead
batch_size = 200
for i in range(0, len(all_chunks), batch_size):
    collection.add(
        ids=ids[i : i + batch_size],
        documents=documents[i : i + batch_size],
        metadatas=metadatas[i : i + batch_size],
    )

print(
    f"Successfully added {collection.count()} chunks to Chroma DB at {vector_db_path}!"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully added 1088 chunks to Chroma DB at e:\Applied AI - iti\Car Dashboard Detection Project\backend\data\vector_store!


In [16]:
import os
import chromadb
from chromadb.utils import embedding_functions
import ollama

# Re-connect to Chroma DB collection
vector_db_path = os.path.abspath("../backend/data/vector_store")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
chroma_client = chromadb.PersistentClient(path=vector_db_path)
collection = chroma_client.get_or_create_collection(
    name="car_manual", embedding_function=embedding_fn
)


def retrieve_context(query, top_k=3):
    results = collection.query(query_texts=[query], n_results=top_k)
    retrieved_texts = results["documents"][0]
    retrieved_metas = results["metadatas"][0]

    context = ""
    sources = []
    for text, meta in zip(retrieved_texts, retrieved_metas):
        context += f"\n--- Source: {meta['source']} ---\n{text}\n"
        sources.append(meta["source"])

    return context, sources


def answer_question(user_query, detected_symbol=None):
    query_with_vision = user_query
    if detected_symbol:
        query_with_vision = f"[Warning Light Detected: {detected_symbol}] {user_query}"

    context, sources = retrieve_context(query_with_vision, top_k=3)

    prompt = f"""You are a helpful car owner assistant. Answer the question using ONLY the provided car manual context below. 
If the information is not in the context, say "I cannot find this in the owner manual." Always cite the manual page numbers provided.

Context:
{context}

User Question: {query_with_vision}

Answer:"""

    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}],
    )

    return {
        "answer": response["message"]["content"],
        "sources": list(set(sources)),
    }

In [13]:
from ultralytics import YOLO

yaml_path = r"E:\Applied AI - iti\Car Dashboard Detection Project\data\images\data.yaml"

yolo_model = YOLO("yolov8n.pt")
results = yolo_model.train(data=yaml_path, epochs=5, imgsz=640, device=0)

best_model_path = os.path.join(results.save_dir, "weights", "best.pt")
print("Training done. Weights saved at:", best_model_path)

Ultralytics 8.4.146  Python-3.13.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Applied AI - iti\Car Dashboard Detection Project\data\images\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.

In [14]:
import os
from ultralytics import YOLO

# Paste the exact path Cell A printed above (or check runs/detect/ folder for the latest train-N)
best_model_path = r"E:\Applied AI - iti\Car Dashboard Detection Project\runs\detect\train-3\weights\best.pt"

trained_yolo = YOLO(best_model_path)

def detect_dashboard_symbol(image_path):
    results = trained_yolo(image_path)
    detected_classes = []
    for box in results[0].boxes:
        cls_id = int(box.cls)
        detected_classes.append(trained_yolo.names[cls_id])
    return list(set(detected_classes))

In [17]:
sample_test_img = r"E:\Applied AI - iti\Car Dashboard Detection Project\data\images\test\images\03_Dash_Warning_Lightss_jpeg_jpg.rf.403197bcbb47bf136b7e7b4085256795.jpg"

if os.path.exists(sample_test_img):
    detected = detect_dashboard_symbol(sample_test_img)
    print("Detected Symbols from Image:", detected)

    # Uses 'Engine Warning' fallback if no detection crosses default threshold
    active_symbol = detected[0] if detected else "Engine Warning"

    fused_result = answer_question(
        user_query="What action should I take right now?",
        detected_symbol=active_symbol,
    )
    print("\n--- FUSED RAG RESPONSE ---")
    print("ANSWER:\n", fused_result["answer"])
    print("\nSOURCES:", fused_result["sources"])
else:
    print(f"Error: Could not find image at path: {sample_test_img}")


image 1/1 E:\Applied AI - iti\Car Dashboard Detection Project\data\images\test\images\03_Dash_Warning_Lightss_jpeg_jpg.rf.403197bcbb47bf136b7e7b4085256795.jpg: 640x640 (no detections), 150.8ms
Speed: 940.3ms preprocess, 150.8ms inference, 4.1ms postprocess per image at shape (1, 3, 640, 640)
Detected Symbols from Image: []

--- FUSED RAG RESPONSE ---
ANSWER:
 According to the owner's manual (Page 370), if the engine oil pressure warning light is on, the recommended action is:

"Indicates that the engine oil pressure is excessively low
 Immediately stop the vehicle in a safe place and contact your Toyota dealer."

So, the action you should take right now is to stop the vehicle in a safe place and contact your Toyota dealer.

SOURCES: ['Manual - Page 469', 'Manual - Page 379', 'Manual - Page 370']
